# Preprocesamiento de Datos y Feature Engineering

## 🎯 Objetivo

En este notebook aprenderemos las técnicas **más importantes** para preparar datos antes de entrenar modelos de Machine Learning.

### 📌 Tema Central

> **"Los modelos son tan buenos como los datos que reciben."**

El 80% del trabajo en un proyecto de ML es **preparación de datos**. Un modelo mediocre con datos bien preparados supera a un modelo sofisticado con datos sucios.

---

## 📚 Contenido

### Parte 1: Preprocesamiento
1. Valores Faltantes
2. Outliers
3. Duplicados

### Parte 2: Transformaciones
4. Encoding de Variables Categóricas
5. Escalamiento
6. Normalización

### Parte 3: Feature Engineering
7. Creación de Features
8. Transformaciones no lineales
9. Interacciones

### Parte 4: Selección de Features
10. Métodos Filter
11. Métodos Wrapper
12. Métodos Embedded

## Parte 1: Limpieza de Datos

### 🧹 1.1 Valores Faltantes (Missing Values)

Los valores faltantes son uno de los problemas más comunes en datos reales.

#### Estrategias:

| Método | Cuándo usarlo | Ventajas | Desventajas |
|--------|----------------|----------|-------------|
| **Eliminar filas** | <5% missing | Simple | Pierde información |
| **Imputar con media/mediana** | Numéricos | Preserva tamaño | Introduce bias |
| **Imputar con moda** | Categóricos | Simple | Puede distorsionar distribución |
| **Imputar con modelo** | >20% missing | Preciso | Computacionalmente costoso |
| **Indicador missing** | Información valiosa | Captura patrón | Aumenta dimensionalidad |

#### 💡 Tips:

* **Analizar patrón**: ¿Missing aleatorio o sistemático?
* **No eliminar automáticamente**: Valores faltantes pueden tener significado
* **Documentar decisiones**: Explicar qué método y por qué

In [0]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Crear dataset con valores faltantes
np.random.seed(42)
df = pd.DataFrame({
    'edad': [25, 30, np.nan, 45, np.nan, 35, 50, np.nan],
    'salario': [50000, 60000, np.nan, 80000, 70000, np.nan, 90000, 75000],
    'ciudad': ['NY', 'LA', np.nan, 'CHI', 'NY', 'LA', np.nan, 'CHI'],
    'compro': [1, 0, 1, 1, 0, np.nan, 1, 0]
})

print("┌──────────────────────────────────────────────────┐")
print("│        DATASET ORIGINAL CON VALORES FALTANTES        │")
print("└──────────────────────────────────────────────────┘")
print(df)
print(f"\n📉 Missing values por columna:")
print(df.isnull().sum())

# Método 1: Imputar con media/mediana
imputer_num = SimpleImputer(strategy='median')
df_imputed = df.copy()
df_imputed[['edad', 'salario']] = imputer_num.fit_transform(df[['edad', 'salario']])

print("\n\n✅ DATASET DESPUÉS DE IMPUTACIÓN (media/mediana):")
print(df_imputed)

## 1.2 Outliers (Valores Atípicos)

### 🔍 Definición

Outliers son observaciones que se desvían significativamente del resto de los datos.

### Métodos de Detección:

#### 1️⃣ **Método IQR (Interquartile Range)**

$$\text{IQR} = Q_3 - Q_1$$

$$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}$$
$$\text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$

#### 2️⃣ **Método Z-Score**

$$z = \frac{x - \mu}{\sigma}$$

Outliers: $|z| > 3$

#### 3️⃣ **Isolation Forest**

Algoritmo de ML para detección de anomalías.

### ⚠️ ¿Qué hacer con outliers?

* **NO eliminar automáticamente** - pueden ser valores válidos
* **Investigar**: ¿Error de medición o fenómeno real?
* **Opciones**: Eliminar, transformar (log, winsorize), usar modelos robustos

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Crear datos con outliers
np.random.seed(42)
data_normal = np.random.normal(100, 15, 95)
outliers = np.array([200, 210, 220, 5, 10])  # Outliers
data = np.concatenate([data_normal, outliers])
df_outliers = pd.DataFrame({'valor': data})

print("┌──────────────────────────────────────────────────┐")
print("│         DETECCIÓN DE OUTLIERS - MÉTODO IQR           │")
print("└──────────────────────────────────────────────────┘")

# Método IQR
Q1 = df_outliers['valor'].quantile(0.25)
Q3 = df_outliers['valor'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower Bound: {lower_bound:.2f}")
print(f"Upper Bound: {upper_bound:.2f}")

outliers_mask = (df_outliers['valor'] < lower_bound) | (df_outliers['valor'] > upper_bound)
print(f"\n🚨 Outliers detectados: {outliers_mask.sum()}")
print(df_outliers[outliers_mask])

## Parte 2: Transformaciones de Datos

### 2.1 Encoding de Variables Categóricas

Los modelos de ML requieren inputs numéricos. Debemos convertir variables categóricas.

| Método | Uso | Ejemplo | Ventajas | Desventajas |
|--------|-----|---------|----------|-------------|
| **Label Encoding** | Ordinales | Tamaño: Pequeño(0), Mediano(1), Grande(2) | Simple | Implica orden |
| **One-Hot Encoding** | Nominales | Color: Rojo[1,0,0], Verde[0,1,0], Azul[0,0,1] | No asume orden | Alta dimensionalidad |
| **Target Encoding** | Alta cardinalidad | Promedio del target por categoría | Reduce dimensionalidad | Puede causar overfitting |

### 2.2 Escalamiento (Scaling)

#### **StandardScaler (Z-score):**

$$x' = \frac{x - \mu}{\sigma}$$

* Media = 0, Desviación estándar = 1
* Usa cuando: Distribución normal, presencia de outliers moderados

#### **MinMaxScaler:**

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

* Rango [0, 1]
* Usa cuando: Necesitas valores acotados, redes neuronales

#### **RobustScaler:**

Usa mediana e IQR (robusto a outliers)

* Usa cuando: Muchos outliers

In [0]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler

# Dataset de ejemplo
df_transform = pd.DataFrame({
    'ciudad': ['NY', 'LA', 'CHI', 'NY', 'LA', 'CHI'],
    'educacion': ['Secundaria', 'Universidad', 'Primaria', 'Universidad', 'Secundaria', 'Universidad'],
    'edad': [25, 45, 30, 50, 28, 35],
    'salario': [50000, 90000, 60000, 95000, 55000, 70000]
})

print("📊 ENCODING - ONE-HOT:")
df_encoded = pd.get_dummies(df_transform, columns=['ciudad'], prefix='ciudad')
print(df_encoded.head())

print("\n📊 ESCALAMIENTO:")
# StandardScaler
scaler_standard = StandardScaler()
df_transform['edad_standard'] = scaler_standard.fit_transform(df_transform[['edad']])

# MinMaxScaler
scaler_minmax = MinMaxScaler()
df_transform['salario_minmax'] = scaler_minmax.fit_transform(df_transform[['salario']])

print(df_transform[['edad', 'edad_standard', 'salario', 'salario_minmax']])

## Parte 3: Feature Engineering

### 🎯 Definición

> **Feature Engineering** es el arte de crear nuevas features a partir de datos existentes para mejorar el performance del modelo.

### 💡 Regla de Oro

**"Features beats algorithms."** - Una feature bien diseñada puede mejorar más el modelo que cambiar de algoritmo.

### Técnicas Comunes:

#### 1️⃣ **Features Polinomiales**

* $x^2, x^3, \sqrt{x}$
* Capturan relaciones no lineales

#### 2️⃣ **Interacciones**

* $x_1 \times x_2$
* Capturan relaciones entre features

#### 3️⃣ **Agregaciones**

* Suma, promedio, max, min por grupo
* Útil en series temporales

#### 4️⃣ **Binning (Discretización)**

* Convertir continuo a categórico
* Edad → Grupos: Joven, Adulto, Mayor

#### 5️⃣ **Fechas**

* Extraer: año, mes, día, día_semana
* Features cíclicas: $\sin(\text{mes}), \cos(\text{mes})$

#### 6️⃣ **Texto**

* Longitud, cantidad de palabras, sentimiento
* TF-IDF, embeddings

In [0]:
# Dataset de ejemplo: Transacciones de e-commerce
df_fe = pd.DataFrame({
    'fecha_compra': pd.to_datetime(['2024-01-15', '2024-02-20', '2024-03-10', '2024-01-22', '2024-02-14']),
    'precio': [50, 150, 30, 200, 75],
    'cantidad': [2, 1, 5, 1, 3],
    'categoria': ['Electrónica', 'Ropa', 'Hogar', 'Electrónica', 'Hogar']
})

print("🔧 FEATURE ENGINEERING - CREACIÓN DE FEATURES:\n")

# 1. Feature de interacción
df_fe['total_compra'] = df_fe['precio'] * df_fe['cantidad']
print("1. Total compra (precio × cantidad):")
print(df_fe[['precio', 'cantidad', 'total_compra']].head())

# 2. Features de fecha
df_fe['mes'] = df_fe['fecha_compra'].dt.month
df_fe['dia_semana'] = df_fe['fecha_compra'].dt.dayofweek
df_fe['trimestre'] = df_fe['fecha_compra'].dt.quarter
print("\n2. Features de fecha:")
print(df_fe[['fecha_compra', 'mes', 'dia_semana', 'trimestre']].head())

# 3. Binning
df_fe['rango_precio'] = pd.cut(df_fe['precio'], bins=[0, 50, 100, 200], labels=['Bajo', 'Medio', 'Alto'])
print("\n3. Binning de precio:")
print(df_fe[['precio', 'rango_precio']].head())

## Parte 4: Selección de Features

### ⚠️ El Problema de Muchas Features

**"Curse of Dimensionality"** - Demasiadas features:
* ❌ Overfitting
* ❌ Lentitud
* ❌ Dificultad de interpretación

### Métodos de Selección:

#### 1️⃣ **Filter Methods**

* Basados en estadísticas (correlación, chi-cuadrado, mutual information)
* Rápidos, independientes del modelo
* Ejemplos: `SelectKBest`, `VarianceThreshold`

#### 2️⃣ **Wrapper Methods**

* Evalúan subconjuntos de features entrenando modelo
* Lentos pero precisos
* Ejemplos: RFE (Recursive Feature Elimination)

#### 3️⃣ **Embedded Methods**

* Selección durante el entrenamiento
* Balance velocidad-precisión
* Ejemplos: Lasso (L1), Random Forest feature importance

### 🎯 Estrategia Recomendada:

1. **Eliminar features de baja varianza**
2. **Eliminar features altamente correlacionadas**
3. **Aplicar método embedded** (ej: Random Forest importance)
4. **Validar con cross-validation**

In [0]:
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
import seaborn as sns
import matplotlib.pyplot as plt

# Crear dataset para clasificación
np.random.seed(42)
X = np.random.randn(100, 10)
y = (X[:, 0] + X[:, 1] > 0).astype(int)  # Target depende solo de 2 features

print("🎯 SELECCIÓN DE FEATURES:\n")

# 1. VarianceThreshold - eliminar features con baja varianza
selector_var = VarianceThreshold(threshold=0.1)
X_var = selector_var.fit_transform(X)
print(f"1. VarianceThreshold: {X.shape[1]} → {X_var.shape[1]} features")

# 2. SelectKBest - top k features por correlación con target
selector_k = SelectKBest(f_classif, k=5)
X_k = selector_k.fit_transform(X, y)
print(f"\n2. SelectKBest (k=5): Top 5 features")
scores = selector_k.scores_
for i, score in enumerate(scores):
    print(f"   Feature {i}: {score:.2f}")

# 3. Random Forest feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)
importances = rf.feature_importances_
print(f"\n3. Random Forest - Feature Importance:")
for i, imp in enumerate(importances):
    print(f"   Feature {i}: {imp:.3f}")

## 📍 Conclusiones y Mejores Prácticas

### 🏆 Key Takeaways

1. **El 80% del trabajo en ML es preparación de datos** - Invértele el tiempo necesario

2. **No hay una estrategia universal** - Cada dataset es diferente

3. **Explora antes de transformar** - Entiende tus datos primero (EDA)

4. **Documenta decisiones** - Por qué imputaste con media, por qué eliminaste outliers

5. **Usa pipelines** - Sklearn Pipelines para reproducibilidad

6. **Evita data leakage** - Fit solo en training, transform en test

### 🚨 Errores Comunes:

* ❌ Escalar antes de train/test split
* ❌ Eliminar outliers sin investigar
* ❌ Crear features usando información del futuro
* ❌ No validar features con cross-validation
* ❌ Sobre-ingeniería: crear demasiadas features sin validar

### ✅ Checklist de Preprocesamiento:

- [ ] Valores faltantes manejados
- [ ] Outliers analizados (eliminar/transformar/mantener)
- [ ] Variables categóricas codificadas
- [ ] Features numéricas escaladas
- [ ] Features creadas y validadas
- [ ] Features seleccionadas
- [ ] Pipeline documentado y reproducible

### 📚 Recursos:

* **Libro**: "Feature Engineering for Machine Learning" (Alice Zheng)
* **Práctica**: Kaggle competitions - estudiar kernels ganadores
* **Herramientas**: scikit-learn, pandas, category_encoders